In [16]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

In [17]:
# Create text sequence
sequence = ["cat", "dog", "cow", "bird",
            "dog", "cow", "bird", "fish",
            "cow", "bird", "fish", "cat",
            "bird", "fish", "cat", "dog"]

print(sequence)

['cat', 'dog', 'cow', 'bird', 'dog', 'cow', 'bird', 'fish', 'cow', 'bird', 'fish', 'cat', 'bird', 'fish', 'cat', 'dog']


In [19]:
# Define window size: 3 words → 1 next word
window_size = 3

# X = input words
# y = next word
X = []
y = []

In [20]:
# Create input-output pairs
for i in range(len(sequence) - window_size):
    X.append(sequence[i:i + window_size])
    y.append(sequence[i + window_size])

print("First 5 input examples:")
print(X[:5])

print("First 5 answers:")
print(y[:5])

First 5 input examples:
[['cat', 'dog', 'cow'], ['dog', 'cow', 'bird'], ['cow', 'bird', 'dog'], ['bird', 'dog', 'cow'], ['dog', 'cow', 'bird']]
First 5 answers:
['bird', 'dog', 'cow', 'bird', 'fish']


In [4]:
# Convert words into numbers
sequence_numbers = np.array([word_to_number[word] for word in sequence])

print(sequence_numbers)

[1 2 3 0 1 2 3 0 1 2 3 0]


In [21]:
# Get unique words
words = sorted(set(sequence))

# Create word-to-number mapping
word_to_number = {word: i for i, word in enumerate(words)}

# Create number-to-word mapping
number_to_word = {i: word for word, i in word_to_number.items()}

print("Word to number:")
print(word_to_number)

Word to number:
{'bird': 0, 'cat': 1, 'cow': 2, 'dog': 3, 'fish': 4}


In [22]:
# Convert X words to numbers
X = np.array([
    [word_to_number[word] for word in row]
    for row in X
])

# Convert y words to numbers
y = np.array([
    word_to_number[word]
    for word in y
])

print("X:")
print(X[:5])

print("y:")
print(y[:5])

X:
[[1 3 2]
 [3 2 0]
 [2 0 3]
 [0 3 2]
 [3 2 0]]
y:
[0 3 2 0 4]


In [23]:
# Reshape X to: samples, timesteps, features
X = X.reshape((X.shape[0], X.shape[1], 1))

print("New X shape:", X.shape)

New X shape: (13, 3, 1)


In [24]:
model = Sequential()

model.add(
    SimpleRNN(
        50,
        activation='relu',
        input_shape=(window_size, 1)
    )
)

# Number of output neurons = number of unique words
model.add(
    Dense(len(words), activation='softmax')
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 50)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           255 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,855 (11.15 KB)

 Trainable params: 2,855 (11.15 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [26]:
model.fit(
    X,
    y,
    epochs=500,
    verbose=2
)

print("Training complete.")

Epoch 1/500
1/1 - 1s - 1s/step - accuracy: 0.3077 - loss: 1.9500
Epoch 2/500
1/1 - 0s - 49ms/step - accuracy: 0.3077 - loss: 1.8991
Epoch 3/500
1/1 - 0s - 54ms/step - accuracy: 0.3077 - loss: 1.8507
Epoch 4/500
1/1 - 0s - 41ms/step - accuracy: 0.3077 - loss: 1.8052
Epoch 5/500
1/1 - 0s - 40ms/step - accuracy: 0.3077 - loss: 1.7626
Epoch 6/500
1/1 - 0s - 40ms/step - accuracy: 0.3077 - loss: 1.7226
Epoch 7/500
1/1 - 0s - 40ms/step - accuracy: 0.3077 - loss: 1.6843
Epoch 8/500
1/1 - 0s - 46ms/step - accuracy: 0.3077 - loss: 1.6474
Epoch 9/500
1/1 - 0s - 42ms/step - accuracy: 0.3077 - loss: 1.6119
Epoch 10/500
1/1 - 0s - 41ms/step - accuracy: 0.3077 - loss: 1.5780
Epoch 11/500
1/1 - 0s - 43ms/step - accuracy: 0.2308 - loss: 1.5459
Epoch 12/500
1/1 - 0s - 40ms/step - accuracy: 0.2308 - loss: 1.5154
Epoch 13/500
1/1 - 0s - 41ms/step - accuracy: 0.1538 - loss: 1.4862
Epoch 14/500
1/1 - 0s - 42ms/step - accuracy: 0.1538 - loss: 1.4589
Epoch 15/500
1/1 - 0s - 43ms/step - accuracy: 0.1538 - loss

In [27]:
# New input
test_input = ["cat", "dog", "cow"]

# Convert words to numbers
test_input = np.array([
    word_to_number[word]
    for word in test_input
])

# RNN expects: samples, timesteps, features
test_input = test_input.reshape(
    (1, window_size, 1)
)

# Predict
predicted = model.predict(
    test_input,
    verbose=0
)

# Get the word with highest probability
predicted_index = np.argmax(predicted[0])

# Convert number back to word
predicted_word = number_to_word[predicted_index]

print("Input:", ["cat", "dog", "cow"])
print("Predicted next word:", predicted_word)

Input: ['cat', 'dog', 'cow']
Predicted next word: bird


In [28]:
predictions = model.predict(X, verbose=0)

predicted_words = []

for prediction in predictions:
    predicted_index = np.argmax(prediction)
    predicted_words.append(
        number_to_word[predicted_index]
    )

print("Predicted words:")
print(predicted_words)

Predicted words:
['bird', 'fish', 'cow', 'bird', 'fish', 'cow', 'bird', 'fish', 'cow', 'bird', 'fish', 'cat', 'bird']
